# Morphological Normalisation at KG Lookup — Proof of Concept

**Research question:** Does normalising test-set token surface forms back to their
canonical form *before* KG lookup close the coverage gap identified in the July rerun?

**Design:** This is an inference-time intervention only. No retraining.
We use the existing July rerun checkpoints and the existing morph pipeline
lookup tables (logainm_lookup.pkl, ud_lookup.pkl) in reverse — as surface→canonical
maps — to ask: if we could match mutated test tokens to their KG entries,
how much coverage would we recover?

**Three measurements:**
1. Baseline token-level KG coverage (exact string match, as per July rerun: ~0.29%)
2. Normalised token-level KG coverage (after reverse lookup)
3. Single-seed F1 delta under normalised lookup injection vs baseline checkpoint

**Inputs required (add as Kaggle datasets):**
- `michaelmarkey64/morph-pipeline-assets`
- `michaelmarkey64/irish-ner-kg-consolidated`
- `michaelmarkey64/iudt-files-adkins`
- `michaelmarkey64/dissertation-rerun-checkpoints-30-07`

In [1]:
# ── Cell 1: Setup (mirrors July rerun Cell group 1) ──────────────────
%pip install -q 'setuptools<81' wheel
%pip install -q pytorch-crf --no-build-isolation
%pip install -q 'seqeval==0.0.10'
%pip install -q transformers datasets stanza scipy

import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'aspell', 'aspell-ga'], check=True)

dicts_output = subprocess.run(['aspell', 'dicts'], capture_output=True, text=True).stdout
assert 'ga' in dicts_output.split(), f'aspell-ga not found: {dicts_output!r}'

import os, json, shutil, logging, sys, pickle
import stanza
import torch
import torch.nn as nn
from functools import partial
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from torchcrf import CRF
from seqeval.metrics import f1_score, precision_score, recall_score
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('norm_poc')

BASE         = '/kaggle/input/datasets/michaelmarkey64'
ASSETS_DIR   = f'{BASE}/morph-pipeline-assets'
KG_DIR       = f'{BASE}/irish-ner-kg-consolidated'
ADKINS_DIR   = f'{BASE}/iudt-files-adkins'
CKPT_DIR     = f'{BASE}/dissertation-rerun-checkpoints-30-07'
WORK_DIR     = '/kaggle/working'

os.makedirs(f'{WORK_DIR}/assets', exist_ok=True)
for fname in os.listdir(ASSETS_DIR):
    shutil.copy(os.path.join(ASSETS_DIR, fname), f'{WORK_DIR}/assets/{fname}')

sys.path.insert(0, f'{WORK_DIR}/assets')
from morph_pipeline import load_pipeline, expand_entity
pipeline = load_pipeline(assets_dir=f'{WORK_DIR}/assets')

log.info('Cell 1 complete: environment ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 21.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Preconfiguring packages ...
Selecting previously unselected package libtext-iconv-perl.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../libtext-iconv-perl_1.7-7build3_amd64.deb ...
Unpacking libtext-iconv-perl (1.7-7build3) ...
Selecting previously unselected package libaspell15:amd64.
Preparing to unpack .../libaspell15_0.60.8-4build1_amd64.deb ...
Unpacking libaspell15:amd64 (0.60.8-4build1) ...
Selecting previously unselected package dictionaries-common.
Preparing to unpack .../dictionaries-common_1.28.14_all.deb ...
Adding 'diversion of /usr/share/dict/words to /usr/share/dict/words.pre-dictionaries-common by dictionaries-common'
Unpacking dictionaries-common (1.28.14) ...
Selecting previously unselected package aspell.
Preparing to unpack .../aspell_0.60.8-4build1_amd64.deb ...
Unpacking aspell (0.60.8-4build1) ...
Selecting previously unselected package aspell-ga.
Preparing to unpack .../aspell-ga_0.50-4-6_all.deb ...
Unpacking asp

2026-08-23 18:39:49,714 INFO Cell 1 complete: environment ready.


Loaded logainm_lookup     : 117 entries
Loaded ud_lookup          : 1792 entries
Loaded manual lexicon     : 69 entries
Loaded wikiann_per        : 191 entities
Loaded wikiann_loc        : 694 entities
aspell-ga available       : True


In [3]:
# ── Cell 2: Load lookup tables and build reverse maps ────────────────
#
# The morph pipeline lookup tables map canonical → surface forms.
# For normalisation we need the inverse: surface form → canonical.
#
# logainm_lookup: {canonical_loc: [attested_surface_forms]}
# ud_lookup:      {lemma: [surface_forms_from_treebank]}
#
# We build a unified surface→canonical reverse map from both.

with open(f'{WORK_DIR}/assets/logainm_lookup.pkl', 'rb') as f:
    logainm_lookup = pickle.load(f)

with open(f'{WORK_DIR}/assets/ud_lookup.pkl', 'rb') as f:
    ud_lookup = pickle.load(f)

# Inspect structure before building reverse map
log.info(f'logainm_lookup: {len(logainm_lookup)} entries')
sample_logainm = list(logainm_lookup.items())[:3]
log.info(f'logainm sample: {sample_logainm}')

log.info(f'ud_lookup: {len(ud_lookup)} entries')
sample_ud = list(ud_lookup.items())[:3]
log.info(f'ud sample: {sample_ud}')

# Build reverse map: surface_form -> canonical_form
# If a surface form maps to multiple canonicals, keep the first encountered
# (logainm takes priority over UD as it is more domain-specific)
surface_to_canonical = {}

def _add_to_reverse(canonical, surfaces):
    """Add surface->canonical mappings, handling both list and set values."""
    if isinstance(surfaces, (list, set, tuple)):
        for s in surfaces:
            if isinstance(s, str) and s not in surface_to_canonical:
                surface_to_canonical[s] = canonical
    elif isinstance(surfaces, str):
        if surfaces not in surface_to_canonical:
            surface_to_canonical[surfaces] = canonical

# Logainm first (higher priority)
for canonical, surfaces in logainm_lookup.items():
    _add_to_reverse(canonical, surfaces)
    # Also add the canonical itself (nominative)
    if canonical not in surface_to_canonical:
        surface_to_canonical[canonical] = canonical

# UD treebank second
for lemma, surfaces in ud_lookup.items():
    _add_to_reverse(lemma, surfaces)
    if lemma not in surface_to_canonical:
        surface_to_canonical[lemma] = lemma

log.info(f'Reverse map built: {len(surface_to_canonical)} surface forms -> canonical')
sample_reverse = [(k, v) for k, v in list(surface_to_canonical.items())[:10]]
log.info(f'Reverse map sample: {sample_reverse}')

2026-08-23 18:40:01,510 INFO logainm_lookup: 117 entries
2026-08-23 18:40:01,512 INFO logainm sample: [('Baile Átha Cliath', {'Baile Átha Cliath', 'Bhaile Átha Cliath'}), ('Ceathrú Ruairí', {'Ceathrú Ruairí', 'Cheathrú Ruairí'}), ('Conamara', {'Conamara', 'Chonamara'})]
2026-08-23 18:40:01,513 INFO ud_lookup: 1792 entries
2026-08-23 18:40:01,514 INFO ud sample: [('Gaeilge', {'nGaeilge', 'Ghaeilge', 'Gaeilge'}), ('Peadar', {'Pheadair', 'Peadar', 'Pheadar'}), ('Bryan', {'Bryan'})]
2026-08-23 18:40:01,516 INFO Reverse map built: 2616 surface forms -> canonical
2026-08-23 18:40:01,517 INFO Reverse map sample: [('Baile Átha Cliath', 'Baile Átha Cliath'), ('Bhaile Átha Cliath', 'Baile Átha Cliath'), ('Ceathrú Ruairí', 'Ceathrú Ruairí'), ('Cheathrú Ruairí', 'Ceathrú Ruairí'), ('Conamara', 'Conamara'), ('Chonamara', 'Conamara'), ('Inis Meáin', 'Inis Meáin'), ('Ghaoth Dobhair', 'Gaoth Dobhair'), ('Gaoth Dobhair', 'Gaoth Dobhair'), ('Gaillimh', 'Gaillimh')]


In [15]:
# ── Cell 3: Load KG entity pool and Adkins test set ──────────────────
import pandas as pd

# Load KG entity pool (canonical forms)
entities_dir = os.path.join(KG_DIR, 'data', 'entities')
pool_files = {'LOC': 'loc_entities.csv', 'ORG': 'org_entities.csv', 'PER': 'per_entities.csv'}
kg_canonical_forms = set()
kg_entity_pool = {}

for ent_type, fname in pool_files.items():
    fpath = os.path.join(entities_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        for _, row in df.iterrows():
            kg_canonical_forms.add(row['entity'])
            kg_entity_pool[row['entity']] = ent_type

log.info(f'KG canonical entity pool: {len(kg_canonical_forms)} entities')

# Load Adkins test set
def load_conll(path):
    sentences, cur = [], []
    with open(path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if cur:
                    sentences.append(cur)
                    cur = []
                continue
            parts = line.split('\t')
            tok, tag = parts[0], parts[-1]
            cur.append((tok, tag))
    if cur:
        sentences.append(cur)
    return sentences

test_path = os.path.join(KG_DIR, 'data', 'conll', 'NER_Irish_test.conll')
if not os.path.exists(test_path):
    # Try alternate path from iudt-files-adkins
    test_path = os.path.join(ADKINS_DIR, 'iudt_test.txt')

adkins_sentences = load_conll(test_path)
log.info(f'Loaded {len(adkins_sentences)} Adkins test sentences')

# Also load label2id from the July rerun
label2id_path = os.path.join(CKPT_DIR, 'label2id.json')
if os.path.exists(label2id_path):
    with open(label2id_path) as f:
        label2id = json.load(f)
else:
    # Reconstruct from test set if not available
    all_tags = sorted({tag for sent in adkins_sentences for _, tag in sent})
    label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {v: k for k, v in label2id.items()}
log.info(f'label2id: {label2id}')

2026-08-23 18:47:20,753 INFO KG canonical entity pool: 1864 entities
2026-08-23 18:47:20,775 INFO Loaded 140 Adkins test sentences
2026-08-23 18:47:20,799 INFO label2id: {'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}


In [16]:
# ── Cell 4: Coverage measurement ─────────────────────────────────────
#
# Measurement 1: Baseline token-level coverage (exact string match)
# Measurement 2: Normalised token-level coverage (after reverse lookup)
#
# A token 'matches' the KG if its surface form (or its normalised form)
# appears in kg_canonical_forms.

baseline_matches = 0
normalised_matches = 0
total_entity_tokens = 0

# Track what normalisation recovers
normalisation_examples = []  # (surface, canonical, kg_entry)
baseline_misses_normalised_hits = []  # tokens missed by exact but caught by normalisation

for sent in adkins_sentences:
    for tok, tag in sent:
        if tag == 'O':
            continue  # only count entity tokens
        total_entity_tokens += 1

        # Measurement 1: exact match
        if tok in kg_canonical_forms:
            baseline_matches += 1

        # Measurement 2: normalised match
        canonical = surface_to_canonical.get(tok, tok)  # fall back to original if not in map
        if canonical in kg_canonical_forms:
            normalised_matches += 1
            if tok not in kg_canonical_forms:
                # This is a recovery — missed by exact, caught by normalisation
                baseline_misses_normalised_hits.append((tok, canonical))
                if len(normalisation_examples) < 20:
                    normalisation_examples.append((tok, canonical))

baseline_coverage = baseline_matches / max(total_entity_tokens, 1)
normalised_coverage = normalised_matches / max(total_entity_tokens, 1)
coverage_gain = normalised_coverage - baseline_coverage

log.info(f'Total entity tokens in test set: {total_entity_tokens}')
log.info(f'Baseline token-level KG coverage (exact): {baseline_matches}/{total_entity_tokens} = {baseline_coverage:.4%}')
log.info(f'Normalised token-level KG coverage:       {normalised_matches}/{total_entity_tokens} = {normalised_coverage:.4%}')
log.info(f'Coverage gain from normalisation:          +{coverage_gain:.4%}')
log.info(f'Tokens recovered by normalisation:         {len(set(t for t,_ in baseline_misses_normalised_hits))}')

log.info('\nNormalisation examples (surface -> canonical -> KG match):')
for surf, canon in normalisation_examples:
    log.info(f'  "{surf}" -> "{canon}" [in KG]')

# Save coverage results
coverage_results = {
    'total_entity_tokens': total_entity_tokens,
    'baseline_matches': baseline_matches,
    'baseline_coverage': baseline_coverage,
    'normalised_matches': normalised_matches,
    'normalised_coverage': normalised_coverage,
    'coverage_gain': coverage_gain,
    'tokens_recovered': len(set(t for t, _ in baseline_misses_normalised_hits)),
    'normalisation_examples': normalisation_examples,
}
with open(f'{WORK_DIR}/normalisation_coverage_results.json', 'w') as f:
    json.dump(coverage_results, f, ensure_ascii=False, indent=2)

log.info('Coverage results saved.')

2026-08-23 18:47:24,632 INFO Total entity tokens in test set: 722
2026-08-23 18:47:24,634 INFO Baseline token-level KG coverage (exact): 199/722 = 27.5623%
2026-08-23 18:47:24,634 INFO Normalised token-level KG coverage:       179/722 = 24.7922%
2026-08-23 18:47:24,635 INFO Coverage gain from normalisation:          +-2.7701%
2026-08-23 18:47:24,637 INFO Tokens recovered by normalisation:         1
2026-08-23 18:47:24,639 INFO 
Normalisation examples (surface -> canonical -> KG match):
2026-08-23 18:47:24,640 INFO   "tAontas" -> "Aontas" [in KG]
2026-08-23 18:47:24,641 INFO   "tAontas" -> "Aontas" [in KG]
2026-08-23 18:47:24,643 INFO Coverage results saved.


In [17]:
# ── Cell 5: BertCRF model definition (identical to July rerun) ───────

MODEL_NAME = 'DCU-NLP/bert-base-irish-cased-v1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class BertCRF(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        emissions = self.linear(self.dropout(outputs.last_hidden_state))
        mask = attention_mask.bool()
        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        return self.crf.decode(emissions, mask=mask)


def tokenize_and_align_labels(sentences, tokenizer, label2id, max_length=256):
    examples = []
    for sent in sentences:
        tokens = [tok for tok, _ in sent]
        tags = [tag for _, tag in sent]
        encoding = tokenizer(tokens, is_split_into_words=True,
                             truncation=True, max_length=max_length)
        word_ids = encoding.word_ids()
        label_ids = [
            label2id[tags[wid]] if wid is not None else label2id['O']
            for wid in word_ids
        ]
        examples.append({
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': label_ids,
            'word_ids': word_ids,
        })
    return examples


class NERDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]


def collate_fn(batch, pad_token_id, pad_label_id):
    max_len = max(len(ex['input_ids']) for ex in batch)
    input_ids, attention_mask, labels, word_ids_batch = [], [], [], []
    for ex in batch:
        pad_len = max_len - len(ex['input_ids'])
        input_ids.append(ex['input_ids'] + [pad_token_id] * pad_len)
        attention_mask.append(ex['attention_mask'] + [0] * pad_len)
        labels.append(ex['labels'] + [pad_label_id] * pad_len)
        word_ids_batch.append(ex['word_ids'] + [None] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
        'word_ids': word_ids_batch,
    }


def make_dataloader(sentences, tokenizer, label2id, batch_size=16, shuffle=False, max_length=256):
    examples = tokenize_and_align_labels(sentences, tokenizer, label2id, max_length=max_length)
    ds = NERDataset(examples)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        collate_fn=partial(collate_fn,
                           pad_token_id=tokenizer.pad_token_id,
                           pad_label_id=label2id['O']),
    )


def evaluate_condition(model, data, label2id):
    id2label_local = {v: k for k, v in label2id.items()}
    all_true, all_pred = [], []
    model.eval()
    with torch.no_grad():
        for batch in data:
            preds = model(batch['input_ids'].cuda(), batch['attention_mask'].cuda())
            for p_seq, true_seq, word_ids in zip(preds, batch['labels'], batch['word_ids']):
                seen_words = set()
                true_words, pred_words = [], []
                for pos, wid in enumerate(word_ids):
                    if wid is None or wid in seen_words or pos >= len(p_seq):
                        continue
                    seen_words.add(wid)
                    true_words.append(id2label_local[int(true_seq[pos])])
                    pred_words.append(id2label_local[int(p_seq[pos])])
                all_true.append(true_words)
                all_pred.append(pred_words)
    return {
        'f1': f1_score(all_true, all_pred),
        'precision': precision_score(all_true, all_pred),
        'recall': recall_score(all_true, all_pred),
    }

log.info('Cell 5 complete: BertCRF and evaluation functions defined.')

2026-08-23 18:47:28,276 INFO HTTP Request: HEAD https://huggingface.co/DCU-NLP/bert-base-irish-cased-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-23 18:47:28,294 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DCU-NLP/bert-base-irish-cased-v1/16cf557550d9f70c100cd0a06e1c4122a9b113ed/config.json "HTTP/1.1 200 OK"
2026-08-23 18:47:28,362 INFO HTTP Request: HEAD https://huggingface.co/DCU-NLP/bert-base-irish-cased-v1/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-23 18:47:28,403 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DCU-NLP/bert-base-irish-cased-v1/16cf557550d9f70c100cd0a06e1c4122a9b113ed/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-23 18:47:28,470 INFO HTTP Request: GET https://huggingface.co/api/models/DCU-NLP/bert-base-irish-cased-v1/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-23 18:47:28,582 INFO HTTP Request: GET htt

In [18]:
# ── Cell 6: Build normalised test set ────────────────────────────────
#
# Apply surface->canonical normalisation to entity tokens in the test set.
# Non-entity tokens (O tag) are left unchanged — we only normalise tokens
# the model has labelled as entities, since those are the ones the KG lookup
# would operate on at inference time.
#
# This produces a modified test set where mutated entity surface forms
# are replaced with their canonical forms. Evaluating on this set with
# the existing baseline checkpoint gives a direct measure of how much
# F1 changes when the alignment bottleneck is removed.
#
# NOTE: gold labels are preserved unchanged. Only the token text changes.
# This is a conservative test — we are asking whether the model, trained
# on canonical forms, performs better when test tokens match that distribution.

def normalise_entity_tokens(sentences, surface_to_canonical):
    """Replace mutated entity surface forms with canonical forms in the test set."""
    normalised = []
    n_replaced = 0
    n_entity_tokens = 0
    for sent in sentences:
        new_sent = []
        for tok, tag in sent:
            if tag != 'O':
                n_entity_tokens += 1
                canonical = surface_to_canonical.get(tok, tok)
                if canonical != tok:
                    n_replaced += 1
                new_sent.append((canonical, tag))
            else:
                new_sent.append((tok, tag))
        normalised.append(new_sent)
    log.info(f'Normalised {n_replaced}/{n_entity_tokens} entity tokens '
             f'({n_replaced/max(n_entity_tokens,1):.1%} replacement rate)')
    return normalised

normalised_test_sentences = normalise_entity_tokens(adkins_sentences, surface_to_canonical)

# Spot-check: show some before/after examples
log.info('\nSpot-check normalisation examples (entity tokens only):')
shown = 0
for orig_sent, norm_sent in zip(adkins_sentences, normalised_test_sentences):
    for (orig_tok, orig_tag), (norm_tok, _) in zip(orig_sent, norm_sent):
        if orig_tag != 'O' and orig_tok != norm_tok:
            log.info(f'  [{orig_tag}] "{orig_tok}" -> "{norm_tok}"')
            shown += 1
            if shown >= 15:
                break
    if shown >= 15:
        break

log.info('Cell 6 complete: normalised test set built.')

2026-08-23 18:47:36,919 INFO Normalised 149/722 entity tokens (20.6% replacement rate)
2026-08-23 18:47:36,920 INFO 
Spot-check normalisation examples (entity tokens only):
2026-08-23 18:47:36,921 INFO   [B-PER] "Sheosamh" -> "Seosamh"
2026-08-23 18:47:36,922 INFO   [B-LOC] "mBaile" -> "Baile"
2026-08-23 18:47:36,923 INFO   [B-PER] "Chaitlín" -> "Caitlín"
2026-08-23 18:47:36,924 INFO   [I-LOC] "Chláir" -> "Clár"
2026-08-23 18:47:36,925 INFO   [B-PER] "Bhreandán" -> "Breandán"
2026-08-23 18:47:36,926 INFO   [I-PER] "hEithir" -> "Eithir"
2026-08-23 18:47:36,926 INFO   [B-PER] "Bhreandán" -> "Breandán"
2026-08-23 18:47:36,927 INFO   [B-PER] "Bhreandáin" -> "Breandáin"
2026-08-23 18:47:36,928 INFO   [B-LOC] "nGaillimh" -> "Gaillimh"
2026-08-23 18:47:36,929 INFO   [I-PER] "Labhráis" -> "Labhrás"
2026-08-23 18:47:36,930 INFO   [I-PER] "Bhrolacháin" -> "Brolachán"
2026-08-23 18:47:36,930 INFO   [B-PER] "Dhearbhla" -> "Dearbhla"
2026-08-23 18:47:36,931 INFO   [I-LOC] "Phárais" -> "Páras"
2026-

In [19]:
# ── Cell 6b: Train baseline seed 42 (single seed for PoC) ────────────
import random, gc
from functools import partial

SEED = 42
EPOCHS = 15
PATIENCE = 3
BATCH_SIZE = 16

# Load training data
train_conll_path = os.path.join(KG_DIR, 'data', 'conll', 'train_final.conll')
val_conll_path = os.path.join(KG_DIR, 'data', 'conll', 'NER_Irish_validation.conll')
train_sentences = load_conll(train_conll_path)
val_sentences = load_conll(val_conll_path)
log.info(f'Loaded {len(train_sentences)} train, {len(val_sentences)} val sentences')

def train_one_seed(train_data, dev_data, label2id, seed, epochs=15, patience=3, lr=3e-5):
    torch.manual_seed(seed)
    random.seed(seed)
    model = BertCRF(MODEL_NAME, num_labels=len(label2id)).cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    best_f1, epochs_no_improve, best_state = -1.0, 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for batch in train_data:
            optimizer.zero_grad()
            loss = model(batch['input_ids'].cuda(),
                        batch['attention_mask'].cuda(),
                        labels=batch['labels'].cuda())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        model.eval()
        result = evaluate_condition(model, dev_data, label2id)
        dev_f1 = result['f1']
        log.info(f'[seed {seed}] epoch {epoch}: loss={epoch_loss/max(n_batches,1):.4f}, dev F1={dev_f1:.4f}')
        if dev_f1 > best_f1:
            best_f1 = dev_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            log.info(f'Early stopping at epoch {epoch}')
            break
    model.load_state_dict(best_state)
    return model, best_f1

train_loader = make_dataloader(train_sentences, tokenizer, label2id,
                               batch_size=BATCH_SIZE, shuffle=True)
dev_loader = make_dataloader(val_sentences, tokenizer, label2id,
                             batch_size=BATCH_SIZE, shuffle=False)

baseline_model, best_dev_f1 = train_one_seed(
    train_loader, dev_loader, label2id, seed=SEED, epochs=EPOCHS, patience=PATIENCE
)
log.info(f'Training complete. Best dev F1: {best_dev_f1:.4f}')

2026-08-23 18:47:40,764 INFO Loaded 1006 train, 100 val sentences
2026-08-23 18:47:41,367 INFO HTTP Request: HEAD https://huggingface.co/DCU-NLP/bert-base-irish-cased-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-23 18:47:41,387 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DCU-NLP/bert-base-irish-cased-v1/16cf557550d9f70c100cd0a06e1c4122a9b113ed/config.json "HTTP/1.1 200 OK"
2026-08-23 18:47:41,450 INFO HTTP Request: HEAD https://huggingface.co/DCU-NLP/bert-base-irish-cased-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-23 18:47:41,469 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DCU-NLP/bert-base-irish-cased-v1/16cf557550d9f70c100cd0a06e1c4122a9b113ed/config.json "HTTP/1.1 200 OK"
2026-08-23 18:47:41,663 INFO HTTP Request: HEAD https://huggingface.co/DCU-NLP/bert-base-irish-cased-v1/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-08-23 18:47:41,730 INFO HTTP Request: GET h

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-23 18:47:41,944 INFO HTTP Request: GET https://huggingface.co/api/models/DCU-NLP/bert-base-irish-cased-v1/discussions?p=0 "HTTP/1.1 200 OK"
BertModel LOAD REPORT from: DCU-NLP/bert-base-irish-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loadi

In [20]:
# ── Cell 7: Evaluate on original and normalised test sets ────────────
# Using model trained in Cell 6b (no checkpoint loading needed)

model = baseline_model
model.eval()

# (a) Evaluate on original test set
orig_loader = make_dataloader(adkins_sentences, tokenizer, label2id, batch_size=16)
orig_result = evaluate_condition(model, orig_loader, label2id)
log.info(f'(a) Original test set F1:    {orig_result["f1"]:.4f} '
         f'[expected ~0.7837 for seed 42 from July rerun]')

# (b) Evaluate on normalised test set
norm_loader = make_dataloader(normalised_test_sentences, tokenizer, label2id, batch_size=16)
norm_result = evaluate_condition(model, norm_loader, label2id)
log.info(f'(b) Normalised test set F1:  {norm_result["f1"]:.4f}')

delta = norm_result['f1'] - orig_result['f1']
log.info(f'\nF1 delta from normalisation: {delta:+.4f}')
log.info(f'Coverage gain:               +{coverage_gain:.4%}')

if delta > 0:
    log.info('RESULT: Normalisation improves F1 — coverage gap is mechanistically confirmed '
             'as the binding constraint. Future work: full multi-seed evaluation.')
elif delta == 0:
    log.info('RESULT: Normalisation produces no change — KG signal is not being used '
             'even when tokens match. The bottleneck may be elsewhere (embedding quality, '
             'injection architecture).')
else:
    log.info('RESULT: Normalisation degrades F1 — canonical forms are out-of-distribution '
             'for the model trained on surface forms. Normalisation at training time '
             'would be needed to match both distributions.')

poc_results = {
    'seed': 42,
    'original_f1': orig_result['f1'],
    'normalised_f1': norm_result['f1'],
    'f1_delta': delta,
    'coverage_baseline': baseline_coverage,
    'coverage_normalised': normalised_coverage,
    'coverage_gain': coverage_gain,
    'interpretation': (
        'positive_delta' if delta > 0
        else 'zero_delta' if delta == 0
        else 'negative_delta'
    ),
}
with open(f'{WORK_DIR}/normalisation_poc_results.json', 'w') as f:
    json.dump(poc_results, f, ensure_ascii=False, indent=2)

log.info('Proof-of-concept complete. Results saved to normalisation_poc_results.json')

2026-08-23 18:50:03,129 INFO (a) Original test set F1:    0.7837 [expected ~0.7837 for seed 42 from July rerun]
2026-08-23 18:50:03,816 INFO (b) Normalised test set F1:  0.7632
2026-08-23 18:50:03,817 INFO 
F1 delta from normalisation: -0.0206
2026-08-23 18:50:03,817 INFO Coverage gain:               +-2.7701%
2026-08-23 18:50:03,818 INFO RESULT: Normalisation degrades F1 — canonical forms are out-of-distribution for the model trained on surface forms. Normalisation at training time would be needed to match both distributions.
2026-08-23 18:50:03,820 INFO Proof-of-concept complete. Results saved to normalisation_poc_results.json


In [ ]:
# ── Cell 8 (optional): Extend to all 7 seeds for distributional evidence ──
#
# Only run this if Cell 7 shows a non-zero delta worth investigating.
# Gives a Wilcoxon-testable set of paired F1 scores.

from scipy.stats import wilcoxon

SEEDS = [13, 42, 123, 456, 789, 2024, 999]
orig_f1s = []
norm_f1s = []

for seed in SEEDS:
    ckpt_path = os.path.join(CKPT_DIR, 'ckpts_baseline', f'seed{seed}.pt')
    if not os.path.exists(ckpt_path):
        log.warning(f'Checkpoint missing for seed {seed}, skipping')
        continue

    model = BertCRF(MODEL_NAME, num_labels=len(label2id)).cuda()
    state = torch.load(ckpt_path, map_location='cuda')
    model.load_state_dict(state, strict=False)
    model.eval()

    orig_loader = make_dataloader(adkins_sentences, tokenizer, label2id, batch_size=16)
    norm_loader = make_dataloader(normalised_test_sentences, tokenizer, label2id, batch_size=16)

    orig_r = evaluate_condition(model, orig_loader, label2id)
    norm_r = evaluate_condition(model, norm_loader, label2id)

    orig_f1s.append(orig_r['f1'])
    norm_f1s.append(norm_r['f1'])

    log.info(f'Seed {seed}: orig={orig_r["f1"]:.4f}, norm={norm_r["f1"]:.4f}, '
             f'delta={norm_r["f1"]-orig_r["f1"]:+.4f}')

    import gc, torch as _torch
    del model
    gc.collect()
    _torch.cuda.empty_cache()

if len(orig_f1s) >= 2:
    stat, p = wilcoxon(norm_f1s, orig_f1s, alternative='greater')
    mean_delta = sum(n - o for n, o in zip(norm_f1s, orig_f1s)) / len(orig_f1s)
    log.info(f'\nAll-seed results:')
    log.info(f'  Original mean F1:    {sum(orig_f1s)/len(orig_f1s):.4f}')
    log.info(f'  Normalised mean F1:  {sum(norm_f1s)/len(norm_f1s):.4f}')
    log.info(f'  Mean delta:          {mean_delta:+.4f}')
    log.info(f'  Wilcoxon p (norm > orig): {p:.4f}')

    all_seed_results = {
        'seeds': SEEDS[:len(orig_f1s)],
        'orig_f1s': orig_f1s,
        'norm_f1s': norm_f1s,
        'mean_orig': sum(orig_f1s)/len(orig_f1s),
        'mean_norm': sum(norm_f1s)/len(norm_f1s),
        'mean_delta': mean_delta,
        'wilcoxon_stat': float(stat),
        'wilcoxon_p': float(p),
        'significant': p < 0.05,
    }
    with open(f'{WORK_DIR}/normalisation_allseed_results.json', 'w') as f:
        json.dump(all_seed_results, f, ensure_ascii=False, indent=2)
    log.info('All-seed results saved.')

In [21]:
# ── Cell 9: Canonicalise training data and train aligned baseline ─────
#
# Replaces every entity surface token in the training set with its
# canonical form, then trains a fresh baseline on this canonicalised
# training data. Evaluates on the already-canonicalised test set
# (normalised_test_sentences from Cell 6).
#
# This tests whether aligning both training and test to canonical form
# closes the F1 gap that inference-time-only normalisation (Cell 7) opened.

def canonicalise_entity_tokens(sentences, surface_to_canonical):
    """Replace mutated entity surface forms with canonical forms."""
    canonicalised = []
    n_replaced = 0
    n_entity_tokens = 0
    for sent in sentences:
        new_sent = []
        for tok, tag in sent:
            if tag != 'O':
                n_entity_tokens += 1
                canonical = surface_to_canonical.get(tok, tok)
                if canonical != tok:
                    n_replaced += 1
                new_sent.append((canonical, tag))
            else:
                new_sent.append((tok, tag))
        canonicalised.append(new_sent)
    log.info(f'Training canonicalisation: replaced {n_replaced}/{n_entity_tokens} '
             f'entity tokens ({n_replaced/max(n_entity_tokens,1):.1%})')
    return canonicalised

# Canonicalise training sentences
canon_train_sentences = canonicalise_entity_tokens(train_sentences, surface_to_canonical)

# Spot-check
log.info('Training canonicalisation spot-check:')
shown = 0
for orig_sent, canon_sent in zip(train_sentences, canon_train_sentences):
    for (orig_tok, orig_tag), (canon_tok, _) in zip(orig_sent, canon_sent):
        if orig_tag != 'O' and orig_tok != canon_tok:
            log.info(f'  [{orig_tag}] "{orig_tok}" -> "{canon_tok}"')
            shown += 1
            if shown >= 10:
                break
    if shown >= 10:
        break

# Train one seed first to validate the approach
SEED = 42
canon_train_loader = make_dataloader(canon_train_sentences, tokenizer, label2id,
                                      batch_size=16, shuffle=True)
dev_loader = make_dataloader(val_sentences, tokenizer, label2id,
                             batch_size=16, shuffle=False)

canon_model, best_dev_f1 = train_one_seed(
    canon_train_loader, dev_loader, label2id,
    seed=SEED, epochs=15, patience=3
)
log.info(f'Canonicalised training complete. Best dev F1: {best_dev_f1:.4f}')

# Evaluate on canonicalised test set (normalised_test_sentences from Cell 6)
canon_test_loader = make_dataloader(normalised_test_sentences, tokenizer, label2id,
                                     batch_size=16)
canon_result = evaluate_condition(canon_model, canon_test_loader, label2id)
log.info(f'Canonicalised train → canonicalised test F1: {canon_result["f1"]:.4f}')

# Compare all three conditions for seed 42
log.info('\n── Seed 42 comparison ──────────────────────────────')
log.info(f'Surface train  → surface test  (July baseline):    0.7837')
log.info(f'Surface train  → canonical test (Cell 7 PoC):      0.7632  (delta -0.0206)')
log.info(f'Canonical train → canonical test (this cell):      {canon_result["f1"]:.4f}  '
         f'(delta {canon_result["f1"] - 0.7837:+.4f} vs July baseline)')

# Save
import json
aligned_results = {
    'seed': SEED,
    'surface_surface_f1': 0.7837,
    'surface_canonical_f1': 0.7632,
    'canonical_canonical_f1': canon_result['f1'],
    'delta_vs_july_baseline': canon_result['f1'] - 0.7837,
}
with open(f'{WORK_DIR}/canonicalised_alignment_results.json', 'w') as f:
    json.dump(aligned_results, f, ensure_ascii=False, indent=2)
log.info('Results saved to canonicalised_alignment_results.json')

2026-08-23 19:10:57,888 INFO Training canonicalisation: replaced 1029/4937 entity tokens (20.8%)
2026-08-23 19:10:57,890 INFO Training canonicalisation spot-check:
2026-08-23 19:10:57,890 INFO   [I-ORG] "Éireann" -> "Éire"
2026-08-23 19:10:57,891 INFO   [B-LOC] "nGaillimh" -> "Gaillimh"
2026-08-23 19:10:57,892 INFO   [B-LOC] "nGaillimh" -> "Gaillimh"
2026-08-23 19:10:57,892 INFO   [I-ORG] "Éireann" -> "Éire"
2026-08-23 19:10:57,893 INFO   [I-PER] "Stáit" -> "stát"
2026-08-23 19:10:57,894 INFO   [I-ORG] "Éireann" -> "Éire"
2026-08-23 19:10:57,894 INFO   [I-ORG] "hÉireann" -> "Éire"
2026-08-23 19:10:57,895 INFO   [I-ORG] "Stáit" -> "stát"
2026-08-23 19:10:57,897 INFO   [B-LOC] "Palaistíne" -> "Palaistín"
2026-08-23 19:10:57,897 INFO   [B-LOC] "gCluain" -> "cluain"
2026-08-23 19:10:58,333 INFO HTTP Request: HEAD https://huggingface.co/DCU-NLP/bert-base-irish-cased-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-23 19:10:58,350 INFO HTTP Request: HEAD https://huggingf

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-23 19:10:58,731 INFO HTTP Request: GET https://huggingface.co/api/models/DCU-NLP/bert-base-irish-cased-v1/discussions?p=0 "HTTP/1.1 200 OK"
BertModel LOAD REPORT from: DCU-NLP/bert-base-irish-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loadi

In [22]:
# ── Verification Cell: Audit reverse map against baseline KG matches ──────────
#
# Checks whether surface_to_canonical converts tokens that already matched
# the KG (baseline matches) into forms that no longer match.
# This confirms or refutes the mechanism behind the coverage loss.

audit_results = {
    'baseline_match_converted_to_nonmatch': [],  # the problematic case
    'baseline_match_unchanged': [],               # map returns same token
    'baseline_match_converted_to_other_match': [],  # converted but still in KG
}

for sent in adkins_sentences:
    for tok, tag in sent:
        if tag == 'O':
            continue
        if tok not in kg_canonical_forms:
            continue  # only looking at baseline matches

        canonical = surface_to_canonical.get(tok, tok)

        if canonical == tok:
            audit_results['baseline_match_unchanged'].append(tok)
        elif canonical in kg_canonical_forms:
            audit_results['baseline_match_converted_to_other_match'].append(
                (tok, canonical)
            )
        else:
            # This is the problematic case: was a match, now is not
            audit_results['baseline_match_converted_to_nonmatch'].append(
                (tok, canonical)
            )

# Summary counts
n_unchanged = len(audit_results['baseline_match_unchanged'])
n_converted_lost = len(audit_results['baseline_match_converted_to_nonmatch'])
n_converted_kept = len(audit_results['baseline_match_converted_to_other_match'])
n_total_baseline = n_unchanged + n_converted_lost + n_converted_kept

log.info(f'Baseline matches audited: {n_total_baseline}')
log.info(f'  Unchanged by map (tok == canonical):     {n_unchanged} ({n_unchanged/n_total_baseline:.1%})')
log.info(f'  Converted, still in KG:                  {n_converted_kept} ({n_converted_kept/n_total_baseline:.1%})')
log.info(f'  Converted, no longer in KG (LOST):       {n_converted_lost} ({n_converted_lost/n_total_baseline:.1%})')

log.info('\nSample of LOST matches (tok -> canonical):')
for tok, canon in audit_results['baseline_match_converted_to_nonmatch'][:20]:
    log.info(f'  "{tok}" -> "{canon}"')

log.info('\nSample of unchanged matches:')
for tok in list(set(audit_results['baseline_match_unchanged']))[:10]:
    log.info(f'  "{tok}"')

# Save
import json
with open(f'{WORK_DIR}/reverse_map_audit.json', 'w') as f:
    json.dump({
        'n_baseline_matches_audited': n_total_baseline,
        'n_unchanged': n_unchanged,
        'n_converted_lost': n_converted_lost,
        'n_converted_kept': n_converted_kept,
        'lost_examples': audit_results['baseline_match_converted_to_nonmatch'][:30],
        'converted_kept_examples': audit_results['baseline_match_converted_to_other_match'][:10],
    }, f, ensure_ascii=False, indent=2)

log.info('Audit saved to reverse_map_audit.json')

2026-08-23 19:24:16,672 INFO Baseline matches audited: 199
2026-08-23 19:24:16,673 INFO   Unchanged by map (tok == canonical):     122 (61.3%)
2026-08-23 19:24:16,674 INFO   Converted, still in KG:                  55 (27.6%)
2026-08-23 19:24:16,675 INFO   Converted, no longer in KG (LOST):       22 (11.1%)
2026-08-23 19:24:16,676 INFO 
Sample of LOST matches (tok -> canonical):
2026-08-23 19:24:16,677 INFO   "Bhreandáin" -> "Breandáin"
2026-08-23 19:24:16,678 INFO   "Dhearbhla" -> "Dearbhla"
2026-08-23 19:24:16,678 INFO   "Árann" -> "Árainn"
2026-08-23 19:24:16,679 INFO   "Mumhan" -> "Mumhain"
2026-08-23 19:24:16,680 INFO   "tAire" -> "aire"
2026-08-23 19:24:16,681 INFO   "gCollchoill" -> "Collchoill"
2026-08-23 19:24:16,681 INFO   "gCollchoill" -> "Collchoill"
2026-08-23 19:24:16,682 INFO   "Mumhan" -> "Mumhain"
2026-08-23 19:24:16,683 INFO   "Mumhan" -> "Mumhain"
2026-08-23 19:24:16,683 INFO   "Mumhan" -> "Mumhain"
2026-08-23 19:24:16,684 INFO   "Bhriain" -> "Brian"
2026-08-23 19:24

In [24]:
# ── KG Canonical Form Audit ───────────────────────────────────────────
#
# Characterises the canonical form conventions actually used in the KG
# entity pool, and checks what proportion of those forms would survive
# the reverse lookup map unchanged.
#
# No model involvement. Pure lookup table vs KG comparison.

import pandas as pd
import pickle, json, re
from collections import Counter

# Clean the KG entity pool — remove NaN and non-string entries
kg_canonical_forms = {e for e in kg_canonical_forms if isinstance(e, str) and e.strip()}
kg_entity_pool = {k: v for k, v in kg_entity_pool.items() if isinstance(k, str) and k.strip()}

log.info(f'KG entity pool after cleaning: {len(kg_canonical_forms)} entities')
# Load KG entity pool (already loaded as kg_canonical_forms and kg_entity_pool)
# Load lookup tables (already loaded as logainm_lookup, ud_lookup, surface_to_canonical)

# ── 1. Capitalisation convention in KG ───────────────────────────────
capitalised = sum(1 for e in kg_canonical_forms if e and e[0].isupper())
lowercased  = sum(1 for e in kg_canonical_forms if e and e[0].islower())
log.info(f'KG entity pool: {len(kg_canonical_forms)} entities')
log.info(f'  Capitalised (proper noun convention): {capitalised} ({capitalised/len(kg_canonical_forms):.1%})')
log.info(f'  Lowercased (common noun convention):  {lowercased} ({lowercased/len(kg_canonical_forms):.1%})')

# ── 2. Mutation prefix presence in KG canonical forms ────────────────
# If KG canonical forms themselves contain mutation markers, the lookup
# tables are not the only source of mismatch.
LENITION_RE  = re.compile(r'^[BCDFGMPS]h', re.UNICODE)
ECLIPSIS_RE  = re.compile(r'^(mb|gc|nd|ng|bhf|bp|dt)', re.IGNORECASE)
TPREFIX_RE   = re.compile(r'^t[A-ZÁÉÍÓÚ]', re.UNICODE)
HPREFIX_RE   = re.compile(r'^h[AEIOUÁÉÍÓÚ]', re.IGNORECASE)

n_lenited   = sum(1 for e in kg_canonical_forms if LENITION_RE.match(e))
n_eclipsed  = sum(1 for e in kg_canonical_forms if ECLIPSIS_RE.match(e))
n_tprefixed = sum(1 for e in kg_canonical_forms if TPREFIX_RE.match(e))
n_hprefixed = sum(1 for e in kg_canonical_forms if HPREFIX_RE.match(e))

log.info(f'\nMutation markers in KG canonical forms:')
log.info(f'  Lenited (Bh/Ch/Dh...):  {n_lenited}')
log.info(f'  Eclipsed (mb/gc/nd...): {n_eclipsed}')
log.info(f'  t-prefixed:             {n_tprefixed}')
log.info(f'  h-prefixed:             {n_hprefixed}')

# ── 3. How many KG canonical forms appear in surface_to_canonical ─────
# i.e. how many KG entries would be altered by the reverse map if
# they appeared as test tokens
kg_in_map          = {e for e in kg_canonical_forms if e in surface_to_canonical}
kg_map_identity    = {e for e in kg_in_map if surface_to_canonical[e] == e}
kg_map_altered     = {e for e in kg_in_map if surface_to_canonical[e] != e}
kg_map_altered_lost = {e for e in kg_map_altered
                       if surface_to_canonical[e] not in kg_canonical_forms}

log.info(f'\nKG canonical forms appearing in reverse map: {len(kg_in_map)}/{len(kg_canonical_forms)} ({len(kg_in_map)/len(kg_canonical_forms):.1%})')
log.info(f'  Map returns same form (identity):    {len(kg_map_identity)}')
log.info(f'  Map alters the form:                 {len(kg_map_altered)}')
log.info(f'  Altered AND result not in KG (risk): {len(kg_map_altered_lost)}')

log.info('\nSample of KG forms the map would alter destructively:')
for e in list(kg_map_altered_lost)[:15]:
    log.info(f'  "{e}" -> "{surface_to_canonical[e]}"')

# ── 4. Entity type breakdown of the 22 lost matches ──────────────────
# Uses kg_entity_pool {entity: type} loaded in Cell 3
lost_by_type = Counter()
for tok, canon in audit_results['baseline_match_converted_to_nonmatch']:
    etype = kg_entity_pool.get(tok, 'UNKNOWN')
    lost_by_type[etype] += 1

log.info(f'\nLost matches by entity type:')
for etype, count in lost_by_type.most_common():
    log.info(f'  {etype}: {count}')

# ── Save ──────────────────────────────────────────────────────────────
kg_audit = {
    'kg_pool_size': len(kg_canonical_forms),
    'capitalised': capitalised,
    'lowercased': lowercased,
    'mutation_markers': {
        'lenited': n_lenited,
        'eclipsed': n_eclipsed,
        't_prefixed': n_tprefixed,
        'h_prefixed': n_hprefixed,
    },
    'kg_forms_in_reverse_map': len(kg_in_map),
    'kg_forms_map_identity': len(kg_map_identity),
    'kg_forms_map_altered': len(kg_map_altered),
    'kg_forms_map_altered_lost': len(kg_map_altered_lost),
    'altered_lost_examples': [(e, surface_to_canonical[e])
                               for e in list(kg_map_altered_lost)[:20]],
    'lost_matches_by_entity_type': dict(lost_by_type),
}
with open(f'{WORK_DIR}/kg_canonical_form_audit.json', 'w') as f:
    json.dump(kg_audit, f, ensure_ascii=False, indent=2)

log.info('KG canonical form audit saved.')

2026-08-23 19:27:43,812 INFO KG entity pool after cleaning: 1863 entities
2026-08-23 19:27:43,815 INFO KG entity pool: 1863 entities
2026-08-23 19:27:43,816 INFO   Capitalised (proper noun convention): 1593 (85.5%)
2026-08-23 19:27:43,817 INFO   Lowercased (common noun convention):  252 (13.5%)
2026-08-23 19:27:43,820 INFO 
Mutation markers in KG canonical forms:
2026-08-23 19:27:43,821 INFO   Lenited (Bh/Ch/Dh...):  248
2026-08-23 19:27:43,822 INFO   Eclipsed (mb/gc/nd...): 151
2026-08-23 19:27:43,822 INFO   t-prefixed:             24
2026-08-23 19:27:43,823 INFO   h-prefixed:             43
2026-08-23 19:27:43,825 INFO 
KG canonical forms appearing in reverse map: 387/1863 (20.8%)
2026-08-23 19:27:43,827 INFO   Map returns same form (identity):    269
2026-08-23 19:27:43,827 INFO   Map alters the form:                 118
2026-08-23 19:27:43,828 INFO   Altered AND result not in KG (risk): 73
2026-08-23 19:27:43,829 INFO 
Sample of KG forms the map would alter destructively:
2026-08-2